In [5]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
from statsmodels.stats.proportion import proportion_confint
sns.set_theme()

In [6]:
df= pd.read_csv(r"C:\Users\riyas\OneDrive\Desktop\Bank_Churn_Analysis\data\raw\Churn_Modelling.csv")
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [7]:
print(df.shape)
print(df[['Age','IsActiveMember','NumOfProducts','Exited']].describe())
print("Missing values:", df.isnull().sum().sum())
print("Duplicate customers:",df['CustomerId'].duplicated().sum())
print("Overall churn rate:", round(df['Exited'].mean()*100,1),"%")

(10000, 14)
                Age  IsActiveMember  NumOfProducts        Exited
count  10000.000000    10000.000000   10000.000000  10000.000000
mean      38.921800        0.515100       1.530200      0.203700
std       10.487806        0.499797       0.581654      0.402769
min       18.000000        0.000000       1.000000      0.000000
25%       32.000000        0.000000       1.000000      0.000000
50%       37.000000        1.000000       1.000000      0.000000
75%       44.000000        1.000000       2.000000      0.000000
max       92.000000        1.000000       4.000000      1.000000
Missing values: 0
Duplicate customers: 0
Overall churn rate: 20.4 %


In [21]:
def churn_summary(mask, label):
    n=int(mask.sum())
    k=int(df.loc[mask,'Exited'].sum())
    lo,hi=proportion_confint(k,n,method='wilson')
    return{'group': label,'n':n,"churned":k,
           'rate_%': round(k/n*100,1),
           'CI_low_%':round(lo*100,1),'CI_high_%' :round(hi*00,1)}
def two_group_test(a_mask, b_mask, a_label, b_label):
    tbl= pd.DataFrame([churn_summary(a_mask,a_label),churn_summary(b_mask,a_label)])
    print(tbl.to_string(index=False))
    ct=[[r.churned,r.n-r.churned] for r in tbl.itertuples()]
    chi2,p,_,_= stats.chi2_contingency(ct)
    print("Chi-square p-value:", "< 0.001" if p < 0.001 else f"{p:.3f}")
    return tbl

#h1 :inactive vs active, churn rate ratio with bootstrap 95% CI
print("---H1: inactive vs active---")
h1= two_group_test(df.IsActiveMember == 0, df.IsActiveMember ==1, 'Inactive', 'Active')
inact= df.loc[df.IsActiveMember ==0, 'Exited'].values
act= df.loc[df.IsActiveMember ==1, 'Exited'].values
rng= np.random.default_rng(42)
boot= [rng.choice(inact,len(inact)).mean()/rng.choice(act,len(act)).mean() for _ in range(5000)]
print("Rate ratio (inactive/active):", round(inact.mean()/act.mean(),2),
      "| 95% CI:", np.percentile(boot, [2.5,97.5]).round(2))

#h2: age 40-59 vs under 40
print("\n--- H2: age 40-59 vs under 40---")
h2= two_group_test(df.Age.between(40,59),df.Age<40, ' Age 40-59','under 40')

#h3 :3+ products vs 1-2 products
h3= two_group_test(df.NumOfProducts >= 3, df.NumOfProducts <=2, '3+ products', '1-2 products')

    
           
    

---H1: inactive vs active---
   group    n  churned  rate_%  CI_low_%  CI_high_%
Inactive 4849     1302    26.9      25.6        0.0
Inactive 5151      735    14.3      13.3        0.0
Chi-square p-value: < 0.001
Rate ratio (inactive/active): 1.88 | 95% CI: [1.74 2.05]

--- H2: age 40-59 vs under 40---
     group    n  churned  rate_%  CI_low_%  CI_high_%
 Age 40-59 3487     1293    37.1      35.5        0.0
 Age 40-59 5987      597    10.0       9.2        0.0
Chi-square p-value: < 0.001
      group    n  churned  rate_%  CI_low_%  CI_high_%
3+ products  326      280    85.9      81.7        0.0
3+ products 9674     1757    18.2      17.4        0.0
Chi-square p-value: < 0.001


In [26]:
df = pd.read_csv(r"C:\Users\riyas\OneDrive\Desktop\Bank_Churn_Analysis\data\raw\Churn_Modelling.csv")
print(df.shape); print(df.dtypes); display(df.head())
baseline = df['Exited'].mean()
print(f'Baseline churn: {baseline:.1%}')

(10000, 14)
RowNumber            int64
CustomerId           int64
Surname                str
CreditScore          int64
Geography              str
Gender                 str
Age                  int64
Tenure               int64
Balance            float64
NumOfProducts        int64
HasCrCard            int64
IsActiveMember       int64
EstimatedSalary    float64
Exited               int64
dtype: object


,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


Baseline churn: 20.4%


In [27]:
for c in ['Geography', 'Gender', 'Tenure', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'Exited']:
    print(c, sorted(df[c].unique()))
print(df[['CreditScore', 'Age', 'Balance', 'EstimatedSalary']].agg(['min', 'max']))

Geography ['France', 'Germany', 'Spain']
Gender ['Female', 'Male']
Tenure [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10)]
NumOfProducts [np.int64(1), np.int64(2), np.int64(3), np.int64(4)]
HasCrCard [np.int64(0), np.int64(1)]
IsActiveMember [np.int64(0), np.int64(1)]
Exited [np.int64(0), np.int64(1)]
     CreditScore  Age    Balance  EstimatedSalary
min          350   18       0.00            11.58
max          850   92  250898.09        199992.48
